In [ ]:
from nn2 import *
i = 64
Rms = lambda*_:Res(RMSNorm(i), *_)
CPA = lambda:(
    Rms(Add(
        Cnv(i, g=2., r=(3,3)),
        Pol(i, h=.5),
        At(i, h=.5, G=2, g=2, r=(12,11)),
        Swi(i, h=4.),
    )),
    Rms(Add(
        Cnv(i, g=2., r=(3,3)),
        Pol(i, h=.5),
        At(i, h=.5, G=2, g=2, r=(12,11)),
        Swi(i, h=4.),
    )),
    Rms(Add(
        Cnv(i, g=2., r=(3,3)),
        Pol(i, h=.5),
        At(i, h=.5, G=2, g=2, r=(12,11)),
        Swi(i, h=4.),
    )),
)
mdl = Seq(
    Zip(
        Embedding(256, i),
        Linear(i, i, 0),
        Linear(i, i, 0)
    ),
    Seq(
        Rms(Swi(i, h=4.)),
        *(_:=lambda n:[
            *CPA(),
            Rms(
                Linear(i, i*2, 0),
                T(MaxPool1d(2)),
                Linear(i*2, i, 0),
                *_(n-1),
                Linear(i, i*2),
                Unflatten(2, (2,-1)),
                Flatten(1, 2)
            ),
            *CPA()
        ]if n else CPA())(7)
    ),
    Swi(i, 4., 256),
).to('cuda',torch.bfloat16)
mdl[1] = torch.compile(mdl[1], fullgraph=True, dynamic=False)
opt = torch.optim.AdamW(mdl.parameters(), fused=True)
lr = torch.optim.lr_scheduler.OneCycleLR(opt, 5e-3, 100000, cycle_momentum=False)
plot = Plot()
for x,t in Dl():
    if lr.last_epoch == lr.total_steps: break
    opt.zero_grad()
    x = mdl[:2](x)
    y = mdl[2](x),
    y += mdl([y[0].argmax(2), x, y[0].softmax(2)@mdl[0][0].weight]),
    l = [functional.cross_entropy(y.mT,t)for y in y]
    reduce(torch.add,l).backward()
    opt.step()
    lr.step()
    plot(l=torch.stack(l).data, a=torch.stack([(y.argmax(2)==t).sum()/t.numel()for y in y]))

In [ ]:
class LogRes(Res):
    def forward(s, _):
        _ = [super().forward(_), _]
        s.log = [_.norm()for _ in _]
        return torch.add(*_)
class LogAdd(Add):
    def forward(s, _):
        _ = [s(_)for s in s]
        s.log = [_.norm()for _ in _]
        return reduce(torch.add, _)
for _ in mdl.modules():
    if isinstance(_, Res):
        _.__class__ = LogRes
    elif isinstance(_, Add):
        _.__class__ = LogAdd
with torch.no_grad():
    mdl[2](mdl[1]._orig_mod(mdl[0](next(Dl())[0])))
plt.plot(torch.tensor([_.log for _ in mdl.modules()if isinstance(_,LogAdd)]).float(), label=list('cpas'))
plt.legend();

In [ ]:
plt.plot(torch.stack([torch.stack(_.log[:-1])/_.log[-1]for _ in mdl.modules()if isinstance(_,LogAdd)]).to('cpu',torch.float), label=list('cpa'))
plt.legend();